In [2]:
# Sprint 50 - Task A: Judge Validation and Finalization

# Builds a human-labeled gold set, scores it with each judge configuration
# proposed this sprint, measures agreement with human labels, and selects one
# final judge configuration per language direction.

# Run order: load gold set -> validate -> score with each config -> compare
# agreement -> select winners -> write the final config doc.

!pip install -q unsloth

import sys
# Add uploaded Kaggle dataset/input folder to Python path
sys.path.append("/kaggle/input/datasets/abdighaz/gepa-modules2")

import pandas as pd
from gold_set import load_gold_set, validate_gold_set
from scoring_harness import score_gold_set_with_all_configs
from agreement_metrics import compare_configs, select_final_config
from config import JUDGE_CONFIGS

pd.set_option('display.max_columns', None)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 MB 20.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 45.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 7.7 MB/s eta 0:00:0

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [3]:
## 1. Load and validate the gold set
# Replace `gold_set_template.csv` with the real 30-50-per-direction reviewed set once labeling is complete.

GOLD_SET_PATH = "/kaggle/input/datasets/abdighaz/gold-set/gold_set_labeled_chatgpt.csv"

gold_set = load_gold_set(GOLD_SET_PATH)
validation = validate_gold_set(gold_set, min_per_direction=30, max_per_direction=50)

print(f"Loaded {validation['n_total']} gold samples")
print("Per-direction counts:", validation['counts'])
if validation['issues']:
    print("\nISSUES FOUND:")
    for issue in validation['issues']:
        print(" -", issue)
else:
    print("No validation issues.")

Loaded 100 gold samples
Per-direction counts: {'EN_CMN': 50, 'EN_YUE': 50}
No validation issues.


In [4]:
## 2. Score the gold set with every proposed judge configuration

assert JUDGE_CONFIGS, "JUDGE_CONFIGS is empty -- wire up this sprint's judge configs in gepa_modules/config.py first."

scored_df = score_gold_set_with_all_configs(JUDGE_CONFIGS, gold_set)
scored_df.head(10)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

,config_name,sample_id,direction,gold_overall,judge_final_score,judge_confidence,judge_flag,n_valid_runs,n_total_runs
0,local_qwen2_gemba_rrwa,CMN_001,EN_CMN,9.8,8.0,1.0,ok,3,3
1,local_qwen2_gemba_rrwa,CMN_002,EN_CMN,9.3,8.0,1.0,ok,3,3
2,local_qwen2_gemba_rrwa,CMN_003,EN_CMN,9.3,7.0,1.0,ok,3,3
3,local_qwen2_gemba_rrwa,CMN_004,EN_CMN,6.7,5.0,1.0,ok,3,3
4,local_qwen2_gemba_rrwa,CMN_005,EN_CMN,9.8,8.0,1.0,ok,3,3
5,local_qwen2_gemba_rrwa,CMN_006,EN_CMN,9.1,8.0,1.0,ok,3,3
6,local_qwen2_gemba_rrwa,CMN_007,EN_CMN,9.3,7.0,1.0,ok,3,3
7,local_qwen2_gemba_rrwa,CMN_008,EN_CMN,10.0,7.0,1.0,ok,3,3
8,local_qwen2_gemba_rrwa,CMN_009,EN_CMN,8.1,7.0,1.0,ok,3,3
9,local_qwen2_gemba_rrwa,CMN_010,EN_CMN,9.8,10.0,1.0,ok,3,3


In [5]:
### 2a. Sanity-check the aggregation flags before trusting the numbers
# Any `judge_failed`, `zero_outliers_excluded`, or `high_disagreement` rows are worth a manual look -- they're exactly the cases the aggregation logic is designed not to silently average away.

flag_counts = scored_df.groupby(['config_name', 'judge_flag']).size().unstack(fill_value=0)
flag_counts

judge_flag,judge_failed,ok
config_name,,
local_qwen2_gemba_rrwa,8,92
local_qwen3_gemba_rrwa,0,100


In [6]:
## 3. Compute agreement with human labels, per config and direction

comparison_df = compare_configs(scored_df)
comparison_df

,config_name,direction,n_total_samples,n_judge_failed,coverage,pearson_r,spearman_rho,mae,weighted_kappa,n
2,local_qwen3_gemba_rrwa,EN_CMN,50,0,1.00,0.546,0.647,0.802,0.549,50
0,local_qwen2_gemba_rrwa,EN_CMN,50,2,0.96,0.489,0.516,2.062,0.261,48
3,local_qwen3_gemba_rrwa,EN_YUE,50,0,1.00,0.550,0.493,1.408,0.513,50
1,local_qwen2_gemba_rrwa,EN_YUE,50,6,0.88,0.336,0.411,3.139,0.165,44


In [7]:
## 4. Select the final config per direction
# Highest Spearman correlation among configs meeting the coverage bar (default 90% of gold samples successfully scored), tie-broken by lowest MAE.

winners_df = select_final_config(comparison_df, min_coverage=0.9)
winners_df

,direction,config_name,n_total_samples,n_judge_failed,coverage,pearson_r,spearman_rho,mae,weighted_kappa,n
0,EN_CMN,local_qwen3_gemba_rrwa,50,0,1.0,0.546,0.647,0.802,0.549,50
1,EN_YUE,local_qwen3_gemba_rrwa,50,0,1.0,0.550,0.493,1.408,0.513,50


In [8]:
## 5. Write the final config doc
# One place documenting the winning config per direction, the agreement numbers that justified 
# it, and the runner-ups for contrast -- replaces the scattered Sprint 46-48 recommendations.

def render_config_doc(winners_df, comparison_df, judge_configs):
    notes_by_name = {c.name: c.notes for c in judge_configs}
    lines = ["# Final Judge Configuration -- Sprint 50 Task A\n"]
    for _, row in winners_df.iterrows():
        direction = row['direction']
        lines.append(f"## {direction}\n")
        lines.append(f"**Selected config:** `{row['config_name']}`\n")
        if notes_by_name.get(row['config_name']):
            lines.append(f"**Notes:** {notes_by_name[row['config_name']]}\n")
        lines.append(
            f"**Agreement with gold set:** Spearman rho={row['spearman_rho']}, "
            f"Pearson r={row['pearson_r']}, MAE={row['mae']}, "
            f"weighted kappa={row['weighted_kappa']}, "
            f"coverage={row['coverage']} (n={row['n']})\n"
        )
        others = comparison_df[
            (comparison_df['direction'] == direction) &
            (comparison_df['config_name'] != row['config_name'])
        ]
        if not others.empty:
            lines.append("**Runner-up configs considered:**\n")
            for _, o in others.iterrows():
                lines.append(
                    f"- `{o['config_name']}`: rho={o['spearman_rho']}, "
                    f"MAE={o['mae']}, coverage={o['coverage']}\n"
                )
        lines.append("")
    return "\n".join(lines)

doc = render_config_doc(winners_df, comparison_df, JUDGE_CONFIGS)
with open("final_judge_config.md", "w") as f:
    f.write(doc)

print(doc)

# Final Judge Configuration -- Sprint 50 Task A

## EN_CMN

**Selected config:** `local_qwen3_gemba_rrwa`

**Notes:** Local Qwen3-4B-Instruct backend

**Agreement with gold set:** Spearman rho=0.647, Pearson r=0.546, MAE=0.802, weighted kappa=0.549, coverage=1.0 (n=50)

**Runner-up configs considered:**

- `local_qwen2_gemba_rrwa`: rho=0.516, MAE=2.062, coverage=0.96


## EN_YUE

**Selected config:** `local_qwen3_gemba_rrwa`

**Notes:** Local Qwen3-4B-Instruct backend

**Agreement with gold set:** Spearman rho=0.493, Pearson r=0.55, MAE=1.408, weighted kappa=0.513, coverage=1.0 (n=50)

**Runner-up configs considered:**

- `local_qwen2_gemba_rrwa`: rho=0.411, MAE=3.139, coverage=0.88


